In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructField, StructType, FloatType, IntegerType, StringType, DataType

import os
from dotenv import load_dotenv

#load environment variables
env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
pgconn = os.getenv('POSTGRE_CONN')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')
driver = os.getenv('PG_DRIVER')
pgjar = os.getenv('POSTGRE_JAR')

In [ ]:
#creating sparksession
spark = SparkSession.\
            builder.\
            appName('teste_dimcolors').\
            config('spark.jars', f'{pgjar}').\
            config('spark.driver.extraClassPath', f'{pgjar}').\
            config('spark.executor.extraClassPath', f'{pgjar}').\
        getOrCreate()
        
spark

In [ ]:
dimcolors = ((spark.\
                read.\
                format('jdbc').\
                option('url', pgconn).\
                option('user', pguser).\
                option('password', pgpwd).\
                option('driver', driver).\
                option('query',
                       
                       """ 
                        select 

                            wrh_c.colorid,
                            wrh_c.colorname,
                            wrh_c._airbyte_ab_id as airbyte_ext_id,
                            wrh_c._airbyte_ab_id as airbyte_hashid,
                            wrh_c._airbyte_emitted_at,
                            wrh_c._airbyte_normalized_at
                        
                        from warehouse.colors as wrh_c
                            left join
                            warehouse.colors_archive as wrh_ac
                        on wrh_c.colorid = wrh_ac.colorid

                       """))).\
            load()

dimcolors.show()

In [ ]:

def selectcolors(spark = spark, connurl = pgconn, dbuser = pguser, dbpwd = pgpwd, dbdriver = driver):
    
    """ 
    
    this function selects data from colors table on warehouse schema
    All parameters are set by default variables that we create at the beginning.
    
    """
       
    dimension_colors = spark.\
                            read.\
                            format('jdbc').\
                            option('url', connurl).\
                            option('user', dbuser).\
                            option('password', dbpwd).\
                            option('driver', dbdriver).\
                            option('query',
                                   
                            """
                                select 

                                    wrh_c.colorid,
                                    wrh_c.colorname,
                                    wrh_c._airbyte_ab_id as airb_extract_hash,
                                    wrh_c._airbyte_ab_id as airbyte_hashid,
                                    wrh_c._airbyte_emitted_at as elt_start,
                                    wrh_c._airbyte_normalized_at as elt_end
                        
                                from warehouse.colors as wrh_c
                                    left join
                                    warehouse.colors_archive as wrh_ac
                                on wrh_c.colorid = wrh_ac.colorid
                                   
                            """).\
                        load()
                            
    return dimension_colors

data = selectcolors()



In [ ]:
def insertcolor(spark = spark, connurl = pgconn, dbuser = pguser, dbpwd = pgpwd, dbdriver = driver, data = data):
    
    data.\
        write.\
        format('jdbc').\
        option('url', connurl).\
        option('user', dbuser).\
        option('dbtable', 'warehouse.testeinsert').\
        option('mode', 'overwrite').\
        option('password', dbpwd).\
        option('driver', dbdriver).\
    save(mode='append')
    
insertcolor()

In [ ]:
#rename column and change date datatype

dimcolors.\
            withColumnRenamed('airbyte_ext_id',  'airb_extract_hash').\
            withColumnRenamed("_airbyte_emitted_at", 'elt_start').\
            withColumnRenamed("_airbyte_normalized_at", 'elt_end')
            
            
print()
